# 02 – Métricas do Lote 1 (L1), por célula

OBS-01, OBS-02, OBS-03, QA-02, QA-03, QA-04 e CAL-01..CAL-04, segundo `spec/metodologia_metricas.md`, **por célula de grelha** (`cell`; `GRID_DEG = 0.01`, cerca de 1 km) em vez de por país.

- Lê `processed/<sensor>_<ano>.parquet` (gerado pelo notebook 01), **um ficheiro de cada vez e só com as colunas necessárias**. Linhas com data ou coordenadas inválidas ficam fora (contam-se em `n_excluded`).
- Guarda resultados parciais por ficheiro em `metrics/_partial_lote1/` e **retoma** se a sessão cair (use `FORCE = True` para refazer). O calendário (`cal_daily`, `cal_monthly`, `cal_annual`) calcula-se **um ano de cada vez** (os anos são independentes) e também fica em parciais por ano.
- Consolida com DuckDB (fora da memória) em pastas Parquet `metrics/l1_*/`, ordenadas por (`ilat`, `ilon`) e particionadas por `series` e `lat_band`; as tabelas por célula são **esparsas** (só células com atividade). O que é igual para todas as células (`partial`, `n_unknown_days`) está em `l1_meta_month_validity` e `l1_meta_year_validity`.
- Escreve `_runs` (grelha, commit, parâmetros, linhas por tabela) **por último**: uma execução que pára a meio não fica registada.
- O código das métricas vem do repo (`git clone` do GitHub público): `data/pipeline/metrics/`. Para atualizar, volte a correr a célula 1.

**Antes de correr tudo:** ajuste os parâmetros da célula 2 e corra a **célula de diagnóstico** (`RUN_DIAGNOSTIC = True`) para um só ano: diz se `cal_monthly` cabe nos 10 GB. Se não couber, mude `GRID_DEG`, `STATS_GRID_DEG` ou `SERIES_PUBLISHED` sem tocar no código.

O atalho `Sprinklers` tem de estar em *O Meu Drive* (ver `data/README.md`). O VIIRS é pesado: se faltar memória, ative *High-RAM* em *Runtime → Change runtime type*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Sprinklers')        # ajustar se o atalho tiver outro nome
PROCESSED = ROOT / 'processed'
METRICS = ROOT / 'metrics'
REPO_URL = 'https://github.com/Rui-Martins31/NASA_Space_Apps_2026_Sprinklers.git'
REPO = Path('/content/repo')


def ensure_repo():
    if (REPO / '.git').exists():
        subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'switch', 'metricas-pipeline'], check=True)
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'metricas-pipeline'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'metricas-pipeline', REPO_URL, str(REPO)], check=True)


ensure_repo()
sys.path.insert(0, str(REPO / 'data' / 'pipeline'))

assert PROCESSED.exists(), f"Nao encontro {PROCESSED}. Corra primeiro o notebook 01 e confirme o atalho 'Sprinklers'."
METRICS.mkdir(exist_ok=True)

In [ ]:
# cell: params
# Fontes do nucleo de calendario. O S-NPP processa-se a parte no notebook 09.
SENSORS = {'modis', 'viirs-jpss1'}
YEARS = None        # None escolhe todos os anos disponiveis para estas fontes.
FORCE = False

# --- grelha e volume: decidir ANTES da corrida completa (ver a celula de diagnostico) ---
GRID_DEG = 0.01              # lado da celula das tabelas de base, em graus (0.01 = ~1 km)
STATS_GRID_DEG = GRID_DEG    # grelha das estatisticas (L3..L7); maior (p. ex. 0.1) se 0.01 nao couber
MIN_ACTIVE_YEARS = 5         # poda das estatisticas: celulas com >= 5 anos validos com atividade
SERIES_PUBLISHED = ('modis', 'viirs', 'combined')   # series de l1_cal_monthly / l1_cal_annual

# --- diagnostico (so corre com RUN_DIAGNOSTIC = True; ver a celula de diagnostico) ---
RUN_DIAGNOSTIC = False
DIAG_YEAR = 2018
DISK_BUDGET_GB = 10

In [ ]:
# cell: defs
import json, re, shutil, subprocess, tempfile, time, datetime
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

from metrics import common
from metrics.counts import (
    KEY_INST, KEY_MONTH, KEY_SAT, count_invalid_coords, obs01_raw_counts, obs02_confidence_dist,
    obs03_filtered_counts, qa02_retention, qa03_cellday_grouped, qa04_redundancy,
)
from metrics.calendar_metrics import (
    active_cells, cal_annual, cal_coverage, cal_daily, cal_monthly, coarsen_cell_days,
    eligible_cell_days, global_day_counts, month_validity, year_validity,
)
from metrics.store import ParquetSink, default_partitions, prepare_table, stats_source, write_run, write_table

common._grid_shape(GRID_DEG)
common._grid_shape(STATS_GRID_DEG)
common.coarsen_cell(np.array([0]), GRID_DEG, STATS_GRID_DEG)     # ValueError se nao for multiplo
if set(SERIES_PUBLISHED) - set(common.SERIES):
    raise ValueError(f'SERIES_PUBLISHED so aceita {common.SERIES}')
STATS_SEPARATE = STATS_GRID_DEG != GRID_DEG

SINK = ParquetSink(METRICS)
COLS = ['instrument', 'satellite', 'latitude', 'longitude', 'acq_date', 'type', 'confidence_class']
FILE_RE = re.compile(r'^(?P<sensor>modis|viirs-[a-z0-9]+)_(?P<year>\d{4})\.parquet$', re.I)
PART = METRICS / '_partial_lote1'
PART.mkdir(exist_ok=True)
ALLOWED_SENSORS = {'modis', 'viirs-jpss1'}
TABLE_NAMES = ['obs01', 'obs02', 'obs03', 'qa02', 'qa03', 'qa04', 'cell_days', 'day_counts']
KEYS = {
    'obs01': KEY_MONTH, 'obs02': KEY_INST + ['confidence_class'], 'obs03': KEY_MONTH,
    'qa02': KEY_MONTH, 'qa03': KEY_SAT, 'qa04': KEY_INST,
}
NAMES = {'obs01': 'l1_obs01_raw_counts', 'obs02': 'l1_obs02_confidence_dist',
         'obs03': 'l1_obs03_filtered_counts', 'qa02': 'l1_qa02_retention',
         'qa03': 'l1_qa03_cellday_grouped', 'qa04': 'l1_qa04_redundancy'}
STATS_MONTHLY = stats_source('l1_cal_monthly', GRID_DEG, STATS_GRID_DEG)
STATS_ANNUAL = stats_source('l1_cal_annual', GRID_DEG, STATS_GRID_DEG)
CAL_TABLES = {'daily': 'l1_cal_daily', 'monthly': 'l1_cal_monthly', 'annual': 'l1_cal_annual'}
if STATS_SEPARATE:
    CAL_TABLES.update(stats_monthly=STATS_MONTHLY, stats_annual=STATS_ANNUAL)


def atomic_part(df, path):
    tmp = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


def atomic_json(value, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, sort_keys=True), encoding='utf-8')
    temporary.replace(path)


def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None


def source_signature(sensor, year, path):
    return dict(schema=3, sensor=sensor, year=year, source=str(path.resolve()),
                size=path.stat().st_size, mtime_ns=path.stat().st_mtime_ns,
                grid_deg=GRID_DEG, confidence=list(common.ELIGIBLE_CONFIDENCE),
                eligible_type=common.ELIGIBLE_TYPE)


def is_done(marker, sensor, year, path):
    info = read_json(marker)
    return (isinstance(info, dict) and info.get('signature') == source_signature(sensor, year, path)
            and all((marker.parent / f'{name}.parquet').is_file() for name in TABLE_NAMES))


def select_sensors():
    selected = ALLOWED_SENSORS if SENSORS is None else set(SENSORS)
    unsupported = selected - ALLOWED_SENSORS
    if unsupported:
        raise ValueError(f'Unsupported core sensors {sorted(unsupported)}. Use 09_har12_snpp.ipynb for S-NPP.')
    return selected


def index_processed():
    found = {}
    for p in sorted(PROCESSED.glob('*.parquet')):
        m = FILE_RE.match(p.name)
        if m:
            found[(m['sensor'].lower(), int(m['year']))] = p
    return found


def xy(df):
    return [pd.to_numeric(df[c], errors='coerce').to_numpy(dtype='float64', na_value=np.nan)
            for c in ('latitude', 'longitude')]


def process_file(sensor, year, path):
    """Calcula as tabelas por ficheiro (por celula) e guarda-as em PART/<sensor>_<ano>/."""
    out = PART / f'{sensor}_{year}'
    out.mkdir(exist_ok=True)
    marker = out / '_done.json'
    marker.unlink(missing_ok=True)
    signature = source_signature(sensor, year, path)
    t0 = time.time()
    df = pd.read_parquet(path, columns=COLS)
    expected = 'MODIS' if sensor == 'modis' else 'VIIRS'
    if not df['instrument'].astype('string').str.upper().eq(expected).fillna(False).all():
        raise ValueError(f'Unexpected instrument in {path}')
    if sensor == 'viirs-jpss1':
        satellite = df['satellite'].astype('string').str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
        if not satellite.isin(['N20', 'NOAA20', 'J1', '1']).all():
            raise ValueError(f'Expected NOAA-20 satellite data in {path}')
    dates = common.parse_dates(df['acq_date'])
    bad_coords = ~common.valid_coords(*xy(df))
    info = dict(sensor=sensor, year=year, grid_deg=GRID_DEG, n_rows=len(df),
                n_invalid_dates=int(dates.isna().sum()),
                n_invalid_coords=count_invalid_coords(df),
                n_excluded=int((dates.isna().to_numpy() | bad_coords).sum()),
                n_rows_outside_file_year=int((dates.dt.year.dropna() != year).sum()))
    tables = {
        'obs01': obs01_raw_counts(df, GRID_DEG), 'obs02': obs02_confidence_dist(df, GRID_DEG),
        'obs03': obs03_filtered_counts(df, GRID_DEG), 'qa02': qa02_retention(df, GRID_DEG),
        'qa03': qa03_cellday_grouped(df, GRID_DEG), 'qa04': qa04_redundancy(df, GRID_DEG),
        'cell_days': eligible_cell_days(df, GRID_DEG), 'day_counts': global_day_counts(df),
    }
    for name, tbl in tables.items():
        if name in KEYS:    # tabelas por celula: colunas de celula e ordem (ilat, ilon)
            tbl = prepare_table(tbl, GRID_DEG, KEYS[name])
        atomic_part(tbl, out / f'{name}.parquet')
    info['seconds'] = round(time.time() - t0, 1)
    info['signature'] = signature
    atomic_json(info, marker)
    return info


def collect(name, dirs):
    grids = {read_json(d / '_done.json').get('grid_deg') for d in dirs}
    assert grids <= {GRID_DEG}, f'parciais calculados com grelhas diferentes {grids}; corra com FORCE = True'
    return pd.concat([pd.read_parquet(d / f'{name}.parquet') for d in dirs], ignore_index=True)


# --- calendario por ano: os anos sao independentes, por isso nunca ha mais de um ano em memoria ---
def year_paths(year):
    return {kind: PART / f'cal_{kind}_{year}.parquet' for kind in CAL_TABLES}


def year_signature(year_keys, files):
    return dict(schema=3, grid_deg=GRID_DEG, stats_grid_deg=STATS_GRID_DEG, series=list(SERIES_PUBLISHED),
                sources=[source_signature(sensor, y, files[(sensor, y)]) for sensor, y in year_keys])


def compute_year(year, cell_days, cov, signature):
    """cal_daily / cal_monthly / cal_annual (e as versoes na grelha das estatisticas) de UM ano."""
    marker = PART / f'cal_{year}.json'
    marker.unlink(missing_ok=True)
    if len(cell_days) and int((cell_days['date'].dt.year != year).sum()):
        raise ValueError(f'{year}: ha celulas-dia fora do ano do ficheiro (ver n_rows_outside_file_year); '
                         'o calendario por ano precisa de ficheiros cujas datas estejam todas no ano')
    daily = cal_daily(cell_days)
    monthly = cal_monthly(cell_days, cov)
    annual = cal_annual(monthly)
    expected = {'modis': int(daily['in_modis'].sum()), 'viirs': int(daily['in_viirs'].sum()),
                'combined': int((daily['in_modis'] | daily['in_viirs']).sum())}
    totals = dict(daily=expected,
                  monthly=monthly.groupby('series')['cell_days'].sum().astype(int).to_dict(),
                  annual=annual.groupby('series')['cell_days'].sum().astype(int).to_dict())
    for series, n in expected.items():
        assert totals['monthly'].get(series, 0) == n, f'{year} {series}: mensal != diaria'
        assert totals['annual'].get(series, 0) == n, f'{year} {series}: anual != mensal'
    if len(monthly):
        wide = monthly.pivot_table(index=['cell', 'year', 'month'], columns='series', values='cell_days', fill_value=0)
        if {'modis', 'viirs', 'combined'} <= set(wide.columns):
            assert (wide['combined'] <= wide['modis'] + wide['viirs']).all(), f'{year}: combined > modis + viirs'
            assert (wide['combined'] >= wide[['modis', 'viirs']].max(axis=1)).all(), f'{year}: combined < max(modis, viirs)'
    monthly = monthly[monthly['series'].isin(SERIES_PUBLISHED)]
    annual = cal_annual(monthly)
    paths = year_paths(year)
    rows = {}
    outputs = {'daily': prepare_table(daily, GRID_DEG, ['date']),
               'monthly': prepare_table(monthly, GRID_DEG, ['year', 'month', 'series']),
               'annual': prepare_table(annual, GRID_DEG, ['year', 'series'])}
    if STATS_SEPARATE:
        coarse = coarsen_cell_days(cell_days, GRID_DEG, STATS_GRID_DEG)
        s_monthly = cal_monthly(coarse, cov)
        s_monthly = s_monthly[s_monthly['series'].isin(SERIES_PUBLISHED)]
        outputs['stats_monthly'] = prepare_table(s_monthly, STATS_GRID_DEG, ['year', 'month', 'series'])
        outputs['stats_annual'] = prepare_table(cal_annual(s_monthly), STATS_GRID_DEG, ['year', 'series'])
    for kind, table in outputs.items():
        atomic_part(table, paths[kind])
        rows[kind] = len(table)
    atomic_json(dict(signature=signature, totals=totals, rows=rows), marker)
    return rows


def year_is_done(year, signature):
    info = read_json(PART / f'cal_{year}.json')
    return (isinstance(info, dict) and info.get('signature') == signature
            and all(p.is_file() for p in year_paths(year).values()))


# --- consolidacao com DuckDB (fora da memoria) ---
def duckdb_list(files):
    return '[' + ', '.join("'" + str(f).replace('\\', '/').replace("'", "''") + "'" for f in files) + ']'


def duckdb_connect():
    con = duckdb.connect()
    con.execute(f"SET temp_directory='{Path(tempfile.gettempdir()).as_posix()}/duckdb_spill'")
    return con


def consolidate(name, files, keys, check_keys=True):
    """Junta os parciais `files` numa tabela publicada (pasta Parquet ordenada por (ilat, ilon) e
    particionada por series/lat_band) e devolve o numero de linhas. Verifica chaves repetidas."""
    source = f'read_parquet({duckdb_list(files)}, union_by_name=true)'
    schema = pd.read_parquet(files[0]).iloc[:0]
    order = (['ilat', 'ilon'] + [k for k in keys if k != 'cell']) if 'cell' in schema.columns else list(keys)
    partitions = default_partitions(schema)
    con = duckdb_connect()
    try:
        n_rows = con.execute(f'SELECT count(*) FROM {source}').fetchone()[0]
        if check_keys and n_rows:
            repeated = con.execute(
                f'SELECT count(*) FROM (SELECT 1 FROM {source} GROUP BY {", ".join(keys)} HAVING count(*) > 1)'
            ).fetchone()[0]
            assert repeated == 0, f'{name}: {repeated} chaves repetidas entre ficheiros (ver AVISO acima)'
        if n_rows == 0:
            SINK.write(name, schema, ())
            return 0
        tmp = METRICS / f'{name}.tmp'
        if tmp.exists():
            shutil.rmtree(tmp)
        if partitions:
            part, target = f', PARTITION_BY ({", ".join(partitions)})', tmp
        else:       # sem particoes o DuckDB escreve um ficheiro: po-lo dentro da pasta da tabela
            tmp.mkdir(parents=True)
            part, target = '', tmp / 'part-0.parquet'
        destination = target.as_posix().replace("'", "''")
        con.execute(f"COPY (SELECT * FROM {source} ORDER BY {', '.join(order)}) TO '{destination}' "
                    f"(FORMAT PARQUET, COMPRESSION zstd{part}, OVERWRITE_OR_IGNORE)")
    finally:
        con.close()
    SINK._publish(tmp, name)
    return int(n_rows)


def scalar(name, expression):
    """Valor de uma agregacao SQL sobre a tabela publicada `name` (le a pasta Parquet com DuckDB)."""
    pattern = (SINK.path(name) / '**' / '*.parquet').as_posix().replace("'", "''")
    con = duckdb_connect()
    try:
        value = con.execute(f"SELECT {expression} FROM read_parquet('{pattern}', hive_partitioning=true, "
                            "union_by_name=true)").fetchone()[0]
    finally:
        con.close()
    return 0 if value is None else value

## Diagnóstico de volume (um só ano) — correr ANTES da corrida completa

Ponha `RUN_DIAGNOSTIC = True` na célula 2 e corra esta célula. Para **um** ano (`DIAG_YEAR`, por defeito 2018) calcula as células-dia e `cal_monthly` à grelha `GRID_DEG` e imprime:

- células ativas (com pelo menos uma célula-dia elegível) e linhas de `cal_monthly` (célula, mês, série);
- células que passam `active_cells` (com atividade em meses válidos; com **um só ano** o limiar usado é 1 ano, por isso é um **majorante** das células que passariam `MIN_ACTIVE_YEARS` anos na corrida completa);
- MB de `cal_daily` e de `cal_monthly` em Parquet (zstd) e a **extrapolação aos anos todos** (nº de anos disponíveis × MB do ano; é aproximada: o VIIRS tem mais deteções que o MODIS), comparada com o orçamento `DISK_BUDGET_GB` (10 GB);
- se `STATS_GRID_DEG` ≠ `GRID_DEG`, o mesmo para a grelha das estatísticas;
- a **sobreposição entre países**: deteções `(cell, data, lat, lon)` que aparecem em mais de um `country`. Antes, `country` fazia parte da chave e a mesma deteção contava uma vez por país; agora conta uma só vez por célula-dia, por isso os totais de `n_eligible`/`cell_days` podem descer nas fronteiras.

Se a extrapolação não couber, mude `GRID_DEG`, `STATS_GRID_DEG` (p. ex. 0.1) ou `SERIES_PUBLISHED` na célula 2 e repita; **não é preciso tocar no código**. Com `RUN_DIAGNOSTIC = False` a célula não faz nada.

In [ ]:
# cell: diagnostic
diag = {}
if not RUN_DIAGNOSTIC:
    print('Diagnostico desligado (RUN_DIAGNOSTIC = False na celula 2).')
else:
    diag_sensors = select_sensors()
    diag_files = index_processed()
    diag_keys = sorted((s, y) for (s, y) in diag_files if s in diag_sensors and y == DIAG_YEAR)
    if not diag_keys:
        raise FileNotFoundError(f'No selected processed files for DIAG_YEAR={DIAG_YEAR} in {PROCESSED}')
    for sensor, y in diag_keys:
        marker = PART / f'{sensor}_{y}' / '_done.json'
        if not (is_done(marker, sensor, y, diag_files[(sensor, y)]) and not FORCE):
            process_file(sensor, y, diag_files[(sensor, y)])
    diag_dirs = [PART / f'{sensor}_{y}' for sensor, y in diag_keys]
    cell_days = collect('cell_days', diag_dirs)
    cov_d = cal_coverage(collect('day_counts', diag_dirs))
    validity_d = month_validity(cov_d)
    daily_d = cal_daily(cell_days)
    monthly_d = cal_monthly(cell_days, cov_d)
    monthly_d = monthly_d[monthly_d['series'].isin(SERIES_PUBLISHED)]
    n_years_data = len({y for (s, y) in diag_files if s in diag_sensors})
    scratch = METRICS / '_diag'
    scratch.mkdir(exist_ok=True)

    def parquet_mb(table, grid, sort_by, name):
        target = scratch / f'{name}.parquet'
        prepare_table(table, grid, sort_by).to_parquet(target, index=False, compression='zstd')
        mb = target.stat().st_size / 1e6
        target.unlink()
        return mb

    mb_daily = parquet_mb(daily_d, GRID_DEG, ['date'], 'cal_daily')
    mb_monthly = parquet_mb(monthly_d, GRID_DEG, ['year', 'month', 'series'], 'cal_monthly')
    diag.update(year=DIAG_YEAR, active_cells=int(cell_days['cell'].nunique()), monthly_rows=len(monthly_d),
                cells_pass_1_valid_year=int(len(active_cells(monthly_d, validity_d, 1))),
                mb_cal_daily=mb_daily, mb_cal_monthly=mb_monthly, n_years=n_years_data,
                est_gb=(mb_daily + mb_monthly) * n_years_data / 1000)
    print(f'[{DIAG_YEAR}] grelha {GRID_DEG} graus, fontes {sorted(diag_sensors)}')
    print(f"celulas ativas: {diag['active_cells']:,}")
    print(f"linhas de cal_monthly (celula, mes, serie): {diag['monthly_rows']:,}")
    print(f"celulas com atividade em meses validos (majorante das que passam MIN_ACTIVE_YEARS={MIN_ACTIVE_YEARS}): "
          f"{diag['cells_pass_1_valid_year']:,}")
    print(f'cal_daily em Parquet: {mb_daily:,.1f} MB | cal_monthly em Parquet: {mb_monthly:,.1f} MB')
    print(f'extrapolacao a {n_years_data} anos (cal_daily + cal_monthly, sem cal_annual/obs/qa): '
          f"{diag['est_gb']:,.1f} GB")
    if STATS_SEPARATE:
        coarse = coarsen_cell_days(cell_days, GRID_DEG, STATS_GRID_DEG)
        s_monthly = cal_monthly(coarse, cov_d)
        s_monthly = s_monthly[s_monthly['series'].isin(SERIES_PUBLISHED)]
        diag['mb_stats_monthly'] = parquet_mb(s_monthly, STATS_GRID_DEG, ['year', 'month', 'series'], 'stats')
        print(f"grelha das estatisticas {STATS_GRID_DEG}: {len(s_monthly):,} linhas, "
              f"{diag['mb_stats_monthly']:,.1f} MB por ano ({diag['mb_stats_monthly'] * n_years_data / 1000:,.1f} GB)")
    verdict = 'CABE' if diag['est_gb'] <= DISK_BUDGET_GB else 'NAO CABE: aumente GRID_DEG/STATS_GRID_DEG ou reduza SERIES_PUBLISHED'
    print(f"comparacao com {DISK_BUDGET_GB} GB: {verdict}")

    # sobreposicao entre paises: a mesma deteccao (celula, dia, lat, lon) em mais de um `country`
    parts = []
    for sensor, y in diag_keys:
        raw = pd.read_parquet(diag_files[(sensor, y)], columns=['country', 'latitude', 'longitude', 'acq_date'])
        lat, lon = xy(raw)
        ok = common.valid_coords(lat, lon)
        raw = raw[ok].assign(latitude=lat[ok], longitude=lon[ok])
        raw['cell'] = common.cell_key(raw['latitude'], raw['longitude'], GRID_DEG)
        parts.append(raw.groupby(['cell', 'acq_date', 'latitude', 'longitude'], observed=True)['country'].nunique())
    n_countries = pd.concat(parts)
    n_multi = int((n_countries > 1).sum())
    diag.update(detections_in_several_countries=n_multi, distinct_detections=int(len(n_countries)))
    print(f'deteccoes (cell, data, lat, lon) em mais de um country: {n_multi:,} de {len(n_countries):,} '
          f'({100 * n_multi / max(len(n_countries), 1):.2f} %)')
    shutil.rmtree(scratch, ignore_errors=True)

In [ ]:
# cell: run
selected_sensors = select_sensors()
files = index_processed()

todo = sorted(k for k in files if k[0] in selected_sensors and (YEARS is None or k[1] in YEARS))
if not todo:
    raise FileNotFoundError(f'No selected processed files in {PROCESSED}')
years = sorted({year for _, year in todo})
if YEARS is not None and set(YEARS) - set(years):
    raise FileNotFoundError(f'Missing selected years: {sorted(set(YEARS) - set(years))}')
print(f'{len(files)} processed files, {len(todo)} selected core files')

for sensor, year in todo:
    marker = PART / f'{sensor}_{year}' / '_done.json'
    if is_done(marker, sensor, year, files[(sensor, year)]) and not FORCE:
        print(f'[reuse] {sensor} {year}')
        continue
    info = process_file(sensor, year, files[(sensor, year)])
    print(info)

selected_dirs = [PART / f'{sensor}_{year}' for sensor, year in todo]
selected_info = [read_json(directory / '_done.json') for directory in selected_dirs]
if not any(info['n_rows'] > info['n_invalid_dates'] for info in selected_info):
    raise ValueError('Selected processed files contain no valid acquisition dates')
log = pd.DataFrame(selected_info).sort_values(['sensor', 'year']).reset_index(drop=True)

# cobertura global (so datas das fontes selecionadas) e calendario, um ano de cada vez
cov = cal_coverage(collect('day_counts', selected_dirs))
cov = cov.loc[cov['date'].dt.year.isin(years)].reset_index(drop=True)
for year in years:
    year_keys = [(sensor, y) for sensor, y in todo if y == year]
    signature = year_signature(year_keys, files)
    if not FORCE and year_is_done(year, signature):
        print(f'[reuse] calendar {year}')
        continue
    compact = pd.concat([pd.read_parquet(PART / f'{sensor}_{y}' / 'cell_days.parquet')
                         for sensor, y in year_keys], ignore_index=True)
    rows = compute_year(year, compact, cov, signature)
    print(f'[calendar] {year}: {len(year_keys)} selected sensors, rows {rows}')
    del compact

In [ ]:
# cell: publish
published = {}
for key, name in NAMES.items():
    published[name] = consolidate(name, [d / f'{key}.parquet' for d in selected_dirs], KEYS[key])
    print(f'{name}: {published[name]} linhas')

CAL_KEYS = {'daily': ['cell', 'date'], 'monthly': ['cell', 'year', 'month', 'series'],
            'annual': ['cell', 'year', 'series'], 'stats_monthly': ['cell', 'year', 'month', 'series'],
            'stats_annual': ['cell', 'year', 'series']}
for kind, name in CAL_TABLES.items():       # os anos nao se sobrepoem: nao ha chaves a verificar
    published[name] = consolidate(name, [year_paths(y)[kind] for y in years], CAL_KEYS[kind], check_keys=False)
    print(f'{name}: {published[name]} linhas')

# tabelas pequenas, sem celula: cobertura global e validade (partial / n_unknown_days)
validity = month_validity(cov)
year_val = year_validity(validity)
for name, tbl in {'l1_cal_coverage': cov, 'l1_meta_month_validity': validity,
                  'l1_meta_year_validity': year_val}.items():
    write_table(SINK, name, tbl, grid_deg=GRID_DEG, partition_by=[])
    published[name] = len(tbl)
    print(f'{name}: {len(tbl)} linhas')

try:
    commit = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ''
# _runs escreve-se POR ULTIMO: uma execucao que parou a meio nao fica registada
write_run(SINK, 'lote1', commit=commit or 'unknown', grid_deg=GRID_DEG, tables=published, params={
    'sensors': sorted(selected_sensors), 'years': years,
    'selected_files': [files[key].name for key in todo],
    'grid_deg': GRID_DEG, 'stats_grid_deg': STATS_GRID_DEG, 'min_active_years': MIN_ACTIVE_YEARS,
    'series_published': list(SERIES_PUBLISHED),
    'eligible_type': common.ELIGIBLE_TYPE, 'eligible_confidence': list(common.ELIGIBLE_CONFIDENCE),
    'sources': [{k: v for k, v in info.items() if k != 'signature'} for info in selected_info],
    'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')})
display(log.drop(columns=['signature'], errors='ignore'))

## Verificação
Confirma as invariantes principais entre as tabelas (falha com `AssertionError` se alguma não bater). O calendário já verificou, ano a ano, que mensal = anual = diária e que `combined` fica entre o máximo e a soma dos sensores.

In [ ]:
# cell: verify
for directory, info in zip(selected_dirs, selected_info):
    o1, o3 = pd.read_parquet(directory / 'obs01.parquet'), pd.read_parquet(directory / 'obs03.parquet')
    assert o1['n_raw'].sum() == info['n_rows'] - info['n_excluded'], f'OBS-01 != linhas validas ({directory.name})'
    assert o1[KEY_MONTH].equals(o3[KEY_MONTH]), f'OBS-03 nao tem os grupos de OBS-01 ({directory.name})'
    q3 = pd.read_parquet(directory / 'qa03.parquet')
    assert (q3['n_grouped'] == q3['n_eligible'] - q3['n_cell_days']).all() and (q3['n_grouped'] >= 0).all()
    q4 = pd.read_parquet(directory / 'qa04.parquet')
    assert (q4['n_redundant'] == q4['sum_cell_days_by_satellite'] - q4['n_cell_days_union']).all()

n_valid = int((log['n_rows'] - log['n_excluded']).sum())
assert scalar('l1_obs01_raw_counts', 'sum(n_raw)') == n_valid, 'OBS-01 publicada != linhas validas'
assert scalar('l1_obs03_filtered_counts', 'sum(n_eligible)') == scalar('l1_qa03_cellday_grouped', 'sum(n_eligible)')
year_totals = [read_json(PART / f'cal_{year}.json')['totals'] for year in years]
published_series = [s for s in common.SERIES if s in SERIES_PUBLISHED]
for series in published_series:
    daily_total = sum(t['daily'][series] for t in year_totals)
    assert scalar('l1_cal_monthly', f"sum(cell_days) FILTER (WHERE series = '{series}')") == daily_total, f'mensal != diaria ({series})'
    assert scalar('l1_cal_annual', f"sum(cell_days) FILTER (WHERE series = '{series}')") == daily_total, f'anual != mensal ({series})'
assert scalar('l1_cal_daily', 'sum(in_modis::INTEGER)') == sum(t['daily']['modis'] for t in year_totals)
assert scalar('l1_cal_daily', 'sum(in_viirs::INTEGER)') == sum(t['daily']['viirs'] for t in year_totals)
assert scalar('l1_cal_daily', 'sum((in_modis OR in_viirs)::INTEGER)') == sum(t['daily']['combined'] for t in year_totals)
assert len(validity) and not validity.duplicated(['series', 'year', 'month']).any()
print('OK: invariantes verificadas')
print('Linhas descartadas por data invalida:', int(log['n_invalid_dates'].sum()),
      '| por coordenadas invalidas:', int(log['n_invalid_coords'].sum()),
      '| total excluido das tabelas por celula:', int(log['n_excluded'].sum()))